In [2]:
import sys, os
# path = os.path.abspath('/gpfs/data1/vclgp/decontot/repos/gedih3/src')
# sys.path.insert(0, path)

In [1]:
import gedih3.gh3driver as gh3

In [2]:
import dask_geopandas
ddf = dask_geopandas.read_parquet('/gpfs/data1/vclgp/decontot/repos/gedih3/tmp/tmp/maryland')
ddf

,geometry,agbd_l4a,datetime,wsci_l4c,pai_z_000_l2b,rh_098_l2a,h3_03
npartitions=47,,,,,,,
,geometry,float32,datetime64[ns],float32,float32,float32,category[known]
,...,...,...,...,...,...,...
...,...,...,...,...,...,...,...
,...,...,...,...,...,...,...
,...,...,...,...,...,...,...


In [3]:
adf = gh3.gh3_aggregate(ddf, target_res=5, agg='mean', columns=['agbd_l4a', 'wsci_l4c'])
adf.head(npartitions=10)

,h3_03,agbd_l4a,wsci_l4c,geometry
h3_05,,,,
852aa913fffffff,832aa9fffffffff,170.073959,9.972333,"POLYGON ((-75.69121 38.32435, -75.78754 38.278..."
852aa94bfffffff,832aa9fffffffff,172.676300,10.052324,"POLYGON ((-76.48681 38.28741, -76.58231 38.240..."
852aa95bfffffff,832aa9fffffffff,201.430206,10.157737,"POLYGON ((-76.39855 38.41757, -76.4944 38.3708..."
852aa983fffffff,832aa9fffffffff,146.488434,9.960100,"POLYGON ((-75.40115 38.46221, -75.49802 38.416..."
852aa98bfffffff,832aa9fffffffff,158.843658,9.944152,"POLYGON ((-75.60075 38.45396, -75.69742 38.407..."


In [6]:
from dask.distributed import Client, progress
client = Client(n_workers=4, threads_per_worker=1, memory_limit='8GB')
client

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:8787/status,
Dashboard: http://127.0.0.1:8787/status,Workers: 4
Total threads: 4,Total memory: 29.80 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:38349,Workers: 0
Dashboard: http://127.0.0.1:8787/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:33315,Total threads: 1
Dashboard: http://127.0.0.1:43999/status,Memory: 7.45 GiB
Nanny: tcp://127.0.0.1:44169,


2025-11-10 13:41:05,257 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle c17447ac91ed6342682bdb4a84943e5f initialized by task ('shuffle-transfer-c17447ac91ed6342682bdb4a84943e5f', 15) executed on worker tcp://127.0.0.1:46817
2025-11-10 13:41:07,010 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle d3befa57143046373b32beb623582478 initialized by task ('shuffle-transfer-d3befa57143046373b32beb623582478', 14) executed on worker tcp://127.0.0.1:38885
2025-11-10 13:41:07,490 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle c17447ac91ed6342682bdb4a84943e5f deactivated due to stimulus 'task-finished-1762800067.486759'
2025-11-10 13:41:07,770 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle d3befa57143046373b32beb623582478 deactivated due to stimulus 'task-finished-1762800067.7662103'
2025-11-10 13:42:11,967 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle 167da44ce1fd1100580cde8d8faacf22 initialized by task ('shuffle-transfer-167da44ce1fd1

In [ ]:
import os
from gedih3.utils import is_parquet

def gh3_part_from_df(df):
    h3_cols = [col for col in df.columns if col.startswith('h3_')]
    return sorted(h3_cols)[0]

def gh3_export_part(df, odir, h3_partition_level=None, fmt='parquet'):
    if df.empty:
        return ''
    
    import h3pandas
    os.makedirs(odir, exist_ok=True)
    
    if h3_partition_level is None:
        h3_partition_level = gh3_part_from_df(df)
    h3parent = df[h3_partition_level].iloc[0]
    opath = os.path.join(odir, f"{h3parent}.{fmt}")
    
    if is_parquet(opath):
        df.to_parquet(opath)
    else:
        df.to_file(opath)
    return opath

In [26]:
df = adf.compute()

In [30]:
df.h3_03.unique()

['832aa9fffffffff', '832aabfffffffff', '832aaefffffffff', '832aa8fffffffff', '832aacfffffffff', '832a85fffffffff', '832aaafffffffff']
Categories (7, object): ['832aa9fffffffff', '832aabfffffffff', '832aaefffffffff', '832aa8fffffffff', '832aacfffffffff', '832a85fffffffff', '832aaafffffffff']

In [21]:
df = adf[adf.h3_03 == '832aa9fffffffff'].compute()
df

,h3_03,agbd_l4a,wsci_l4c,geometry
h3_05,,,,
852aa913fffffff,832aa9fffffffff,170.073959,9.972333,"POLYGON ((-75.69121 38.32435, -75.78754 38.278..."
852aa94bfffffff,832aa9fffffffff,172.676300,10.052324,"POLYGON ((-76.48681 38.28741, -76.58231 38.240..."
852aa95bfffffff,832aa9fffffffff,201.430206,10.157737,"POLYGON ((-76.39855 38.41757, -76.4944 38.3708..."
852aa983fffffff,832aa9fffffffff,146.488434,9.960100,"POLYGON ((-75.40115 38.46221, -75.49802 38.416..."
852aa98bfffffff,832aa9fffffffff,158.843658,9.944152,"POLYGON ((-75.60075 38.45396, -75.69742 38.407..."
852aa98ffffffff,832aa9fffffffff,239.462982,10.406340,"POLYGON ((-75.49216 38.33275, -75.58868 38.286..."
852aa99bfffffff,832aa9fffffffff,144.614120,9.787276,"POLYGON ((-75.50979 38.58373, -75.60681 38.537..."
852aa9c7fffffff,832aa9fffffffff,198.280563,10.170441,"POLYGON ((-75.80029 38.44537, -75.89676 38.399..."
852aa9d7fffffff,832aa9fffffffff,163.928040,9.790902,"POLYGON ((-75.70989 38.57528, -75.8067 38.5290..."


In [22]:
gh3_export_part(df, odir)

'/gpfs/data1/vclgp/decontot/repos/gedih3/tmp/tmp/md_agg/832aa9fffffffff.parquet'

In [33]:
adf

,h3_03,agbd_l4a,wsci_l4c,geometry
npartitions=47,,,,
,object,object,object,geometry
,...,...,...,...
...,...,...,...,...
,...,...,...,...
,...,...,...,...


In [35]:
h3_col = 'h3_03'
adf[h3_col] = adf[h3_col].astype(str)
adf[h3_col].dtype

dtype('O')

In [36]:
import pandas as pd
odir = '/gpfs/data1/vclgp/decontot/repos/gedih3/tmp/tmp/md_agg'

write_task = adf.groupby(h3_col, observed=True).apply(gh3.gh3_export_part,
                            odir=odir,
                            fmt='parquet',
                            include_groups=False,
                            meta=str,
                            )

write_task = write_task.persist()
progress(write_task)


VBox()

2025-11-10 13:54:00,931 - distributed.worker - ERROR - Compute Failed
Key:       ('operation-d4415f456e8cf353465c7de4de0183ba', 43)
State:     executing
Task:  <Task ('operation-d4415f456e8cf353465c7de4de0183ba', 43) apply_and_enforce(..., ...)>
Exception: "IndexError('list index out of range')"
Traceback: '  File "/gpfs/data1/vclgp/decontot/environments/gh3_dev/lib/python3.13/site-packages/dask/dataframe/core.py", line 98, in apply_and_enforce\n    df = func(*args, **kwargs)\n  File "/gpfs/data1/vclgp/decontot/environments/gh3_dev/lib/python3.13/site-packages/dask/dataframe/dask_expr/_groupby.py", line 1199, in operation\n    return dask_func(\n        frame,\n    ...<6 lines>...\n        **kwargs,\n    )\n  File "/gpfs/data1/vclgp/decontot/environments/gh3_dev/lib/python3.13/site-packages/dask/dataframe/dask_expr/_groupby.py", line 1235, in groupby_slice_apply\n    return _groupby_slice_apply(\n        df,\n    ...<7 lines>...\n        **kwargs,\n    )\n  File "/gpfs/data1/vclgp/deco

In [ ]:
import geopandas as gpd
world = gpd.read_file(gpd.datasets.get_path('naturalearth_lowres'))


In [ ]:
import os
import dask.dataframe
import dask_geopandas
from dask.distributed import Client

tmp = '/gpfs/data1/vclgp/decontot/temp/dask-tmp'
client = Client(n_workers=10, threads_per_worker=1, memory_limit='30GB', local_directory=tmp)
client#.shutdown()  # Uncomment to stop the client

In [ ]:
from shapely.geometry import box 
region = box(-95.0, 25.0, -65.0, 50.0)

In [ ]:
import gedih3.gh3driver as gh3

In [ ]:
cols = ['shot_number', 'elev_lowestmode_l2a', 'wsci_l4c', 'agbd_l4a', 'h3_03']

In [ ]:
q = 'wsci_l4c >= 0 and agbd_l4a > 0'
h3dir = '/gpfs/data1/vclgp/data/iss_gedi/h3_mock/database/'

gh3_df = gh3.gh3_load(region=region, columns=cols, query=q, gh3_dir=h3dir)
gh3_df

In [ ]:
import pandas as pd

# agg = {'wsci_l4c': 'mean', 'elev_lowestmode_l2a': 'mean', 'agbd_l4a': ['mean', 'std']}
agg = lambda x: pd.Series((x.wsci_l4c / x.agbd_l4a).mean())
res = 5

adf = gh3.gh3_aggregate(gh3_df, target_res=res, agg=agg)
adf

In [ ]:
import glob
h3dir = '/gpfs/data1/vclgp/data/iss_gedi/h3_mock/database/'
h3dirs = glob.glob(os.path.join(h3dir, 'h3_*/'))

In [ ]:
import geopandas as gpd
from gedih3.gh3driver import intersect_h3_geometries, gh3_read_meta

def _load_map(d, columns=None):
    files = glob.glob(os.path.join(d, '**/*.parquet'), recursive=True)
    return gpd.read_parquet(files, columns=columns)

def gh3_load_from_map(columns=None, region=None, query=None, gh3_dir=h3dir):
    h3_part = gh3_read_meta("h3_partition_level", gh3_root_dir=gh3_dir)
    h3_part_col = f"h3_{h3_part:02d}"
    h3_ids = gh3_read_meta("h3_partition_ids", gh3_root_dir=gh3_dir)
    
    out_cols = None
    if columns is not None:
        if h3_part_col not in columns:
            columns.append(h3_part_col)
            
        if 'geometry' not in columns:
            columns.append('geometry')

        out_cols = columns.copy()
        
        if query is not None:
            available_cols = gh3_read_meta("h3_columns", gh3_root_dir=gh3_dir)
            q_cols = [col for col in available_cols if col in query]
            columns = list(set(columns + q_cols))        

    if region is not None:
        h3_ids = intersect_h3_geometries(region, h3_ids=h3_ids)
        h3_dirs = [os.path.join(gh3_dir, f"{h3_part_col}={i}") for i in h3_ids]
    else:
        h3_dirs = glob.glob(os.path.join(gh3_dir, f"{h3_part_col}=*/"))

    _meta = _load_map(h3_dirs[0], columns=columns)
    ddf = dask.dataframe.from_map(_load_map, h3_dirs, columns=columns, meta=_meta)
    ddf = dask_geopandas.from_dask_dataframe(ddf, geometry='geometry')

    if query is not None:
        ddf = ddf.query(query)
        if out_cols is not None:
            ddf = ddf[out_cols]

    return ddf

ddf = gh3_load_from_map(region=region, columns=cols, query=q, gh3_dir=h3dir)
ddf

In [ ]:
from gedih3.gh3driver import gh3_aggregate_func, gh3_part_from_df, gh3_add_geometry
from gedih3.h3utils import fix_h3_geometry

def gh3_aggregate_from_map(gh3_df, target_res=5, agg='mean', columns=None, query=None, add_geometry=True):
    _meta = gh3_aggregate_func(df=gh3_df._meta, res=target_res, agg=agg, cols=columns)

    if query is not None:
        gh3_df = gh3_df.query(query)

    agg_df = gh3_df.map_partitions(gh3_aggregate_func, res=target_res, agg=agg, cols=columns, meta=_meta)
    agg_df = agg_df.set_index(f"h3_{target_res:02d}", sort=False)
    
    if add_geometry:
        _gmeta = gpd.GeoDataFrame(columns=agg_df._meta.columns.tolist() + ['geometry'], geometry='geometry', crs=4326)
        agg_df = agg_df.map_partitions(gh3_add_geometry, meta=_gmeta)

    return agg_df

adf = gh3_aggregate_from_map(ddf, target_res=5, agg=agg, add_geometry=True)
adf

In [ ]:
import gedih3 as gh3
tmp = '//gpfs/data1/vclgp/data/iss_gedi/h3_mock/tmp/duckdb'
ddb = gh3.sqlutils.init_duckdb(temp_directory=tmp)

In [ ]:
def data_spec(hex_id=None, year=None):
    db_path = '/gpfs/data1/vclgp/data/iss_gedi/h3_mock/database'
    h3_part = "*"
    year_part = "*"
    if hex_id is not None:
        h3_part = f"{hex_id}"
    if year is not None:
        year_part = f"year={year}"
    return f"{db_path}/{h3_part}/{year_part}/*.parquet"

In [ ]:
ddb.sql(f"""
    SELECT *
    FROM read_parquet('{data_spec()}')
    WHERE  {q}
    LIMIT 10
""")